# 📊 Ecommerce Dataset Visualization
## Vietnamese E-commerce Products Analysis

**Dataset**: Merged E-commerce Dataset (12 sources, 47,530 products)

In [ ]:
# Install required libraries
!pip install pandas matplotlib seaborn plotly kaleido -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ Libraries imported successfully!')

In [ ]:
# Load the dataset from Google Drive or local upload
from google.colab import files

print('📁 Upload Merged_Ecommerce_Dataset.csv')
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Reading: {filename}')
    df = pd.read_csv(filename, encoding='utf-8-sig')

print(f'\n✅ Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

In [ ]:
# Basic Dataset Info
print('=' * 60)
print('📋 DATASET OVERVIEW')
print('=' * 60)
print(f'\nTotal Products: {df.shape[0]:,}')
print(f'Total Columns: {df.shape[1]}')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print('\n' + '=' * 60)
print('📊 COLUMN DATA TYPES')
print('=' * 60)
print(df.dtypes)

In [ ]:
# Sample data preview
print('=' * 60)
print('👀 SAMPLE DATA')
print('=' * 60)
df.head(3)

In [ ]:
# Products by Source
print('=' * 60)
print('🏪 PRODUCTS BY SOURCE')
print('=' * 60)

source_counts = df['source'].value_counts().sort_values(ascending=False)
print(source_counts)

In [ ]:
# 1. Bar Chart - Products by Source
fig, ax = plt.subplots(figsize=(14, 6))

colors = sns.color_palette('husl', len(source_counts))
bars = ax.barh(source_counts.index[::-1], source_counts.values[::-1], color=colors[::-1])

ax.set_xlabel('Number of Products', fontsize=12)
ax.set_ylabel('Source', fontsize=12)
ax.set_title('📦 Products by E-commerce Source', fontsize=14, fontweight='bold')

# Add value labels
for bar, val in zip(bars, source_counts.values[::-1]):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2, f'{val:,}', 
            va='center', fontsize=10)

plt.tight_layout()
plt.savefig('01_products_by_source.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: 01_products_by_source.png')

In [ ]:
# 2. Pie Chart - Market Share
fig, ax = plt.subplots(figsize=(10, 10))

explode = [0.02] * len(source_counts)
colors = sns.color_palette('husl', len(source_counts))

wedges, texts, autotexts = ax.pie(source_counts.values, 
                                   labels=source_counts.index,
                                   autopct='%1.1f%%',
                                   explode=explode,
                                   colors=colors,
                                   shadow=True,
                                   startangle=90)

ax.set_title('🥧 Market Share by E-commerce Source', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('02_market_share_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: 02_market_share_pie.png')

In [ ]:
# 3. Price Distribution by Source
# Convert price to numeric
df['price_numeric'] = pd.to_numeric(df['price'], errors='coerce')

print('=' * 60)
print('💰 PRICE STATISTICS')
print('=' * 60)
print(df.groupby('source')['price_numeric'].describe().round(0))

In [ ]:
# 4. Box Plot - Price Distribution by Source
fig, ax = plt.subplots(figsize=(14, 8))

# Filter for better visualization (prices < 50,000,000 VND)
df_price = df[df['price_numeric'] < 50000000].copy()

order = source_counts.index.tolist()

sns.boxplot(data=df_price, x='source', y='price_numeric', 
            order=order, palette='husl', ax=ax)

ax.set_xlabel('Source', fontsize=12)
ax.set_ylabel('Price (VND)', fontsize=12)
ax.set_title('💵 Price Distribution by E-commerce Source', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)

# Format y-axis
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('03_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: 03_price_distribution.png')

In [ ]:
# 5. Top Categories by Source
print('=' * 60)
print('📂 TOP CATEGORIES')
print('=' * 60)

top_categories = df.groupby('source')['category_name'].value_counts()\
    .groupby('source').head(5)\
    .reset_index(name='count')

print(top_categories.pivot(index='category_name', columns='source', values='count').head(10))

In [ ]:
# 6. Heatmap - Categories vs Sources
fig, ax = plt.subplots(figsize=(16, 10))

# Get top 15 categories overall
top_15_cats = df['category_name'].value_counts().head(15).index.tolist()
df_top = df[df['category_name'].isin(top_15_cats)]

# Create pivot table
pivot = pd.crosstab(df_top['category_name'], df_top['source'])

sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax, 
            cbar_kws={'label': 'Number of Products'})

ax.set_xlabel('Source', fontsize=12)
ax.set_ylabel('Category', fontsize=12)
ax.set_title('🔥 Category Distribution Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('04_category_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: 04_category_heatmap.png')

In [ ]:
# 7. Average Price by Source (Bar Chart)
avg_price = df.groupby('source')['price_numeric'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))

colors = sns.color_palette('viridis', len(avg_price))
bars = ax.bar(avg_price.index, avg_price.values, color=colors)

ax.set_xlabel('Source', fontsize=12)
ax.set_ylabel('Average Price (VND)', fontsize=12)
ax.set_title('💰 Average Product Price by Source', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)

# Format y-axis
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('05_avg_price.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: 05_avg_price.png')

In [ ]:
# 8. Stock Distribution
df['stock_numeric'] = pd.to_numeric(df['stock'], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stock by source
avg_stock = df.groupby('source')['stock_numeric'].mean().sort_values(ascending=False)

axes[0].bar(avg_stock.index, avg_stock.values, color=sns.color_palette('coolwarm', len(avg_stock)))
axes[0].set_xlabel('Source')
axes[0].set_ylabel('Average Stock')
axes[0].set_title('📦 Average Stock by Source')
axes[0].tick_params(axis='x', rotation=45)

# Stock histogram
df_stock = df[df['stock_numeric'].notna() & (df['stock_numeric'] < 500)]
axes[1].hist(df_stock['stock_numeric'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Stock Quantity')
axes[1].set_ylabel('Frequency')
axes[1].set_title('📊 Stock Distribution (Products with <500 stock)')

plt.tight_layout()
plt.savefig('06_stock_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: 06_stock_distribution.png')

In [ ]:
# 9. Top 20 Brands
print('=' * 60)
print('🏷️ TOP 20 BRANDS')
print('=' * 60)

top_brands = df['brand'].value_counts().head(20)
print(top_brands)

In [ ]:
# 10. Top Brands Visualization
fig, ax = plt.subplots(figsize=(12, 8))

colors = sns.color_palette('Spectral', 20)
bars = ax.barh(top_brands.index[::-1], top_brands.values[::-1], color=colors[::-1])

ax.set_xlabel('Number of Products')
ax.set_ylabel('Brand')
ax.set_title('🏆 Top 20 Brands by Product Count', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('07_top_brands.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: 07_top_brands.png')

In [ ]:
# 11. Rating Distribution
df['rating_numeric'] = pd.to_numeric(df['rating'], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating by source
avg_rating = df.groupby('source')['rating_numeric'].mean().sort_values(ascending=False)

axes[0].bar(avg_rating.index, avg_rating.values, 
            color=sns.color_palette('RdYlGn', len(avg_rating)))
axes[0].set_xlabel('Source')
axes[0].set_ylabel('Average Rating')
axes[0].set_title('⭐ Average Rating by Source')
axes[0].set_ylim(0, 5)
axes[0].tick_params(axis='x', rotation=45)

# Rating histogram
df_rating = df[df['rating_numeric'].notna()]
axes[1].hist(df_rating['rating_numeric'], bins=20, color='gold', edgecolor='orange')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Frequency')
axes[1].set_title('📈 Rating Distribution')

plt.tight_layout()
plt.savefig('08_rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: 08_rating_distribution.png')

In [ ]:
# 12. Reviews Count Analysis
df['reviews_numeric'] = pd.to_numeric(df['reviews_count'], errors='coerce')

top_reviewed = df.nlargest(10, 'reviews_numeric')[['product_name', 'source', 'rating', 'reviews_count', 'price']]
print('=' * 60)
print('🔥 TOP 10 MOST REVIEWED PRODUCTS')
print('=' * 60)
print(top_reviewed)

In [ ]:
# 13. Data Quality Analysis
print('=' * 60)
print('✅ DATA QUALITY CHECK')
print('=' * 60)

quality_data = []
for col in df.columns:
    non_null = df[col].notna().sum()
    pct = (non_null / len(df)) * 100
    quality_data.append({'Column': col, 'Non-Null': non_null, 'Completeness %': pct})

quality_df = pd.DataFrame(quality_data).sort_values('Completeness %', ascending=False)
print(quality_df.to_string(index=False))

In [ ]:
# 14. Data Completeness Bar Chart
fig, ax = plt.subplots(figsize=(12, 8))

quality_sorted = quality_df.sort_values('Completeness %')
colors = ['#2ecc71' if x > 80 else '#f39c12' if x > 50 else '#e74c3c' 
          for x in quality_sorted['Completeness %']]

ax.barh(quality_sorted['Column'], quality_sorted['Completeness %'], color=colors)
ax.set_xlabel('Completeness (%)')
ax.set_ylabel('Column')
ax.set_title('📋 Data Completeness by Column', fontsize=14, fontweight='bold')
ax.axvline(x=80, color='green', linestyle='--', alpha=0.7, label='80% threshold')
ax.legend()

plt.tight_layout()
plt.savefig('09_data_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: 09_data_quality.png')

In [ ]:
# 15. Interactive Plotly Chart - Products by Source
fig = px.bar(
    source_counts.reset_index(),
    x='source',
    y='count',
    color='count',
    color_continuous_scale='Viridis',
    title='📊 Interactive: Products by E-commerce Source',
    labels={'source': 'Source', 'count': 'Number of Products'}
)

fig.update_layout(
    xaxis_tickangle=-45,
    template='plotly_white'
)

fig.show()
fig.write_html('10_interactive_products.html')
print('✅ Interactive chart saved: 10_interactive_products.html')

In [ ]:
# 16. Interactive Treemap - Categories
category_counts = df['category_name'].value_counts().head(30).reset_index()
category_counts.columns = ['category', 'count']

fig = px.treemap(
    category_counts,
    path=['category'],
    values='count',
    title='🌳 Top 30 Categories Treemap',
    color='count',
    color_continuous_scale='RdYlBu'
)

fig.show()
fig.write_html('11_interactive_treemap.html')
print('✅ Interactive treemap saved: 11_interactive_treemap.html')

In [ ]:
# 17. Summary Statistics Dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Total Products by Source', 'Avg Price by Source',
                    'Avg Rating by Source', 'Avg Stock by Source'),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

# Products
fig.add_trace(go.Bar(x=source_counts.index, y=source_counts.values, 
                    marker_color='steelblue'), row=1, col=1)

# Price
fig.add_trace(go.Bar(x=avg_price.index, y=avg_price.values, 
                    marker_color='forestgreen'), row=1, col=2)

# Rating
fig.add_trace(go.Bar(x=avg_rating.index, y=avg_rating.values, 
                    marker_color='gold'), row=2, col=1)

# Stock
fig.add_trace(go.Bar(x=avg_stock.index, y=avg_stock.values, 
                    marker_color='coral'), row=2, col=2)

fig.update_layout(height=800, showlegend=False, 
                  title_text='📊 E-commerce Dataset Summary Dashboard')

fig.show()
fig.write_html('12_summary_dashboard.html')
print('✅ Dashboard saved: 12_summary_dashboard.html')

In [ ]:
# Download all charts
from google.colab import files
import os

print('📥 Downloading all charts...')
for f in os.listdir('.'):
    if f.endswith('.png') or f.endswith('.html'):
        files.download(f)
        print(f'  - {f}')

print('\n✅ All files downloaded!')

---
## 📝 Summary

### Key Findings:
- **Total Products**: 47,530 across 12 e-commerce sources
- **Top Source**: Canifa with 9,539 products (20.1%)
- **Price Range**: Varies significantly by source and category
- **Data Quality**: Most columns have >80% completeness

### Visualizations Created:
1. Products by Source (Bar Chart)
2. Market Share (Pie Chart)
3. Price Distribution (Box Plot)
4. Category Heatmap
5. Average Price by Source
6. Stock Distribution
7. Top 20 Brands
8. Rating Distribution
9. Data Quality Analysis
10-12. Interactive Plotly Charts

**Dataset**: Merged_Ecommerce_Dataset.csv
**Sources**: Concung, Emart, Guardian, NhaSachPhuongNam, SuperSports, Tiki, Canifa, Hòa Phát, Nội Thất, Nhà Xinh, TGDD, Yody